# ADS Bibliometric Analysis Pipeline

This notebook is the single entry point for the `ads_bib` package.
It executes all steps sequentially, with configuration cells between steps
so that downstream parameters can be set based on upstream results.

**Pipeline Steps:**
1. Search & Export — Query ADS API, resolve bibcodes to metadata
2. Translation — Detect languages, translate non-English titles/abstracts
3. Tokenization — Create full_text, tokenize with spaCy
4. AND — Author Name Disambiguation (placeholder)
5. Topic Modeling & Curation — BERTopic + datamapplot + cluster removal
6. Citation Networks — Direct, Co-Citation, Bibliographic Coupling, Author Co-Citation

---
## Setup

In [1]:
import os
from pathlib import Path

# Initialize paths (relative to this notebook's location)
from ads_bib.config import init_paths, load_env
from ads_bib._utils.costs import CostTracker

paths = init_paths()  # uses CWD = notebook directory
load_env()

# Cost tracker for all API calls (OpenRouter, etc.)
tracker = CostTracker()

# Shared OpenRouter pricing mode (used in translation, embeddings, topic labeling)
OPENROUTER_COST_MODE = "hybrid"  # 'hybrid', 'strict', 'fast'

# Verify
print(f"Project root: {paths['project_root']}")
print(f"Data dir:     {paths['data']}")

# If you see a spaCy version warning later, update the model:
# !python -m spacy download en_core_web_lg

Project root: c:\Users\rapha\Documents\Studium\Promotionsstudium\MPIWG\2_Notebooks\ADS_Pipeline
Data dir:     c:\Users\rapha\Documents\Studium\Promotionsstudium\MPIWG\2_Notebooks\ADS_Pipeline\data


---
# Phase 1: Search & Export

Query the NASA ADS API for publications matching a search string,
then resolve all bibcodes (publications + references) into full metadata.

### 1.1 Search Configuration

In [ ]:
# --- SEARCH CONFIGURATION ---
# Modify the query string for your research question.
# See: https://ui.adsabs.harvard.edu/help/search/search-syntax

ADS_TOKEN = os.getenv("ADS_API_KEY")

# Example: Einstein's GR papers + citation expansion + free-text search
Set_A = "docs(library/P0hyiLw0T8qsyHSBpWigJA)"  # Einstein GR library
Set_B = f"citations({Set_A}) AND year:1915-2000"   # 1-hop citations
Set_C = f"citations(citations({Set_A})) AND year:1915-2000"  # 2-hop
Set_D = f"({Set_A}) OR ({Set_B}) OR ({Set_C})"
Set_T = "(docs(library/_-RCjJotRKCZC1PP03MqwA) AND year:1915-2000)"  # Treder library

gr_terms = [
    '(general AND relativi*)', 'gravit*', '(allgemein* AND relativi*)',
    '"relativité générale"', '"teoria della relatività"',
    '"Gravité quantique"', '"Gravità quantistica"',
    '(einheitlich* AND feld*)', 'Quantengravitation', '"champ unifié"',
    '(unified AND field*)', '"quantum gravity"', 'cosmo*', 'Kosmo*',
]
Set_E = f"abs:({' OR '.join(gr_terms)}) AND year:1911-2000"
QUERY = f"author:\"Treder, H*\""   # f"({Set_D}) OR ({Set_T}) OR ({Set_E})"

# Set to False to load the latest cached results instead of re-fetching
FORCE_REFRESH = False

### 1.2 Execute Search

In [ ]:
from ads_bib.search import search_ads, save_search_results
from ads_bib._utils.io import load_pickle

latest = paths["raw"] / "search_results_latest.pkl"

if FORCE_REFRESH or not latest.exists():
    bibcodes, references, esources, fulltext_urls = search_ads(QUERY, ADS_TOKEN)
    save_search_results((bibcodes, references, esources, fulltext_urls), paths["raw"])
else:
    print(f"Loading cached results: {latest.name}")
    bibcodes, references, esources, fulltext_urls = load_pickle(latest)

print(f"\nBibcodes:        {len(bibcodes):,}")
print(f"Unique refs:     {len(set(r for rl in references for r in rl)):,}")
print(f"Total refs:      {sum(len(rl) for rl in references):,}")

### 1.3 Export & Resolve Metadata

In [ ]:
from ads_bib.export import resolve_dataset
from ads_bib._utils.io import save_json_lines, load_json_lines

pubs_path = paths["raw"] / "publications.json"
refs_path = paths["raw"] / "references.json"

if FORCE_REFRESH or not pubs_path.exists():
    publications, refs = resolve_dataset(
        bibcodes, references, esources, fulltext_urls, ADS_TOKEN
    )
    save_json_lines(publications, pubs_path)
    save_json_lines(refs, refs_path)
else:
    print("Loading cached exports ...")
    publications = load_json_lines(pubs_path)
    refs = load_json_lines(refs_path)

print(f"\nPublications: {len(publications):,}")
print(f"References:   {len(refs):,}")
publications.info()

In [ ]:
publications.head(10)

---
# Phase 2: Translation

Detect non-English titles and abstracts with fasttext,
then translate them using either OpenRouter (API) or a local HuggingFace model.

### 2.1 Translation Configuration

In [ ]:
# --- TRANSLATION CONFIGURATION ---
# Provider options: 'openrouter' (API) or 'huggingface' (local, with/without CUDA)

TRANSLATION_PROVIDER = "openrouter"  # or "huggingface"
TRANSLATION_MODEL = "google/gemini-3-flash-preview"  # OpenRouter model, or "google/translategemma-4b-it" for HF
TRANSLATION_API_KEY = os.getenv("OPENROUTER_API_KEY")
TRANSLATION_MAX_WORKERS = 50  # parallel workers for API-based translation
# OPENROUTER_COST_MODE is defined in the setup cell above

# Path to fasttext language ID model (download from https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.bin)
FASTTEXT_MODEL = paths["models"] / "lid.176.bin"


### 2.2 Language Detection

In [ ]:
from ads_bib.translate import detect_languages

print("=== Publications ===")
publications = detect_languages(publications, ["Title", "Abstract"], model_path=FASTTEXT_MODEL)

print("\n=== References ===")
refs = detect_languages(refs, ["Title", "Abstract"], model_path=FASTTEXT_MODEL)

### 2.3 Translate Non-English Entries

In [ ]:
from ads_bib.translate import translate_dataframe

print("=== Translating Publications ===")
publications, cost_pubs = translate_dataframe(
    publications, ["Title", "Abstract"],
    provider=TRANSLATION_PROVIDER,
    model=TRANSLATION_MODEL,
    api_key=TRANSLATION_API_KEY,
    max_workers=TRANSLATION_MAX_WORKERS,
    openrouter_cost_mode=OPENROUTER_COST_MODE,
    cost_tracker=tracker,
)

print("\n=== Translating References ===")
refs, cost_refs = translate_dataframe(
    refs, ["Title", "Abstract"],
    provider=TRANSLATION_PROVIDER,
    model=TRANSLATION_MODEL,
    api_key=TRANSLATION_API_KEY,
    max_workers=TRANSLATION_MAX_WORKERS,
    openrouter_cost_mode=OPENROUTER_COST_MODE,
    cost_tracker=tracker,
)

pub_cost = cost_pubs.get("cost_usd")
ref_cost = cost_refs.get("cost_usd")
print("\nTranslation Cost Snapshot:")
print(f"  Publications: {'n/a' if pub_cost is None else f'${pub_cost:.4f}'}")
print(f"  References:   {'n/a' if ref_cost is None else f'${ref_cost:.4f}'}")


---
# Phase 3: Tokenization

Create `full_text` (Title + Abstract) and tokenize with spaCy.

In [ ]:
from ads_bib.tokenize import tokenize_texts

publications = tokenize_texts(publications)
refs = tokenize_texts(refs)

### Checkpoint: Save Phase 1-3 Results

In [ ]:
from ads_bib._utils.io import save_json_lines

save_json_lines(publications, paths["processed"] / "publications_translated_tokenized.json")
save_json_lines(refs, paths["processed"] / "references_translated_tokenized.json")
print("Checkpoint saved.")

---
# Phase 4: Author Name Disambiguation (Placeholder)

This step will use an external AND package once it's ready.
For now, the pipeline continues without disambiguation.

In [ ]:
# === AND PLACEHOLDER ===
# Uncomment when the AND package is available:
#
# from and_package import disambiguate
# publications = disambiguate(publications)
# refs = disambiguate(refs)

print("AND step skipped (placeholder). Continuing without disambiguation.")

---
# Phase 5: Topic Modeling & Curation

Use BERTopic to discover thematic clusters, visualize with datamapplot,
then remove unwanted clusters to curate the dataset.

**Important:** Set parameters below based on your dataset size from Phase 1.

### 5.1 Embedding Configuration

In [2]:
# --- EMBEDDING CONFIGURATION ---
# Providers: 'local', 'huggingface_api', 'openrouter'

EMBEDDING_PROVIDER = "openrouter"
EMBEDDING_MODEL = "google/gemini-embedding-001"
EMBEDDING_API_KEY = os.getenv("OPENROUTER_API_KEY")  # only needed for API providers

# For testing: set SAMPLE_SIZE to an integer (e.g. 200). None = full dataset.
SAMPLE_SIZE = None

### 5.2 Compute Embeddings

In [5]:
from ads_bib._utils.io import load_json_lines

publications = load_json_lines(paths["processed"] / "publications_translated_tokenized.json")
refs = load_json_lines(paths["processed"] / "references_translated_tokenized.json")
print(f"Loaded {len(publications):,} publications")
print(f"Loaded {len(refs):,} references")

Loaded 382 publications
Loaded 499 references


In [6]:
from ads_bib.topic_model import compute_embeddings

df = publications.copy()
if SAMPLE_SIZE is not None:
    df = df.sample(n=min(SAMPLE_SIZE, len(df)), random_state=42).reset_index(drop=True)
    print(f"SAMPLING: {len(df):,} documents")

documents = df["full_text"].tolist()

embeddings = compute_embeddings(
    documents,
    provider=EMBEDDING_PROVIDER,
    model=EMBEDDING_MODEL,
    cache_dir=paths["embeddings_cache"],
    api_key=EMBEDDING_API_KEY,
    openrouter_cost_mode=OPENROUTER_COST_MODE,
    cost_tracker=tracker,
)
print(f"Embeddings shape: {embeddings.shape}")

  Loaded embeddings from cache: embeddings_openrouter_google_gemini-embedding-001.npz
Embeddings shape: (382, 3072)


### 5.3 Dimensionality Reduction Configuration

In [7]:
# --- DIMENSIONALITY REDUCTION ---
# Methods: 'pacmap' (fast, good) or 'umap' (slower, density-preserving with densmap)

DIM_REDUCTION_METHOD = "pacmap"

# PaCMAP parameters
PARAMS_5D = dict(n_neighbors=60, MN_ratio=0.5, FP_ratio=1.0, random_state=42)
PARAMS_2D = dict(n_neighbors=60, MN_ratio=0.5, FP_ratio=1.0, random_state=42)

# Cache suffix for uniqueness
model_safe = EMBEDDING_MODEL.replace('/', '_')
CACHE_SUFFIX = f"{EMBEDDING_PROVIDER}_{model_safe}_{DIM_REDUCTION_METHOD}"

In [8]:
from ads_bib.topic_model import reduce_dimensions

reduced_5d, reduced_2d = reduce_dimensions(
    embeddings,
    method=DIM_REDUCTION_METHOD,
    params_5d=PARAMS_5D,
    params_2d=PARAMS_2D,
    cache_dir=paths["dim_reduction_cache"],
    cache_suffix=CACHE_SUFFIX,
)

  Loaded 5d_openrouter_google_gemini-embedding-001_pacmap from cache
  Loaded 2d_openrouter_google_gemini-embedding-001_pacmap from cache
  5D shape: (382, 5), 2D shape: (382, 2)


### 5.4 Clustering Configuration

**Adjust `min_cluster_size` based on your dataset size!**
- ~180k docs → `min_cluster_size=180` (~0.1%)
- ~50k docs → `min_cluster_size=100`
- ~5k docs → `min_cluster_size=25`
- Testing (200 docs) → `min_cluster_size=10`

In [9]:
# --- TOPIC BACKEND + CLUSTERING CONFIGURATION ---
TOPIC_BACKEND = "toponymy_evoc"  # 'bertopic', 'toponymy', 'toponymy_evoc'
CLUSTERING_METHOD = "fast_hdbscan"  # for BERTopic only: 'fast_hdbscan' or 'hdbscan'

n_docs = len(df)
MIN_CLUSTER_SIZE = max(10, int(n_docs * 0.001))  # ~0.1% of dataset
print(f"Dataset: {n_docs:,} documents → min_cluster_size={MIN_CLUSTER_SIZE}")

CLUSTER_PARAMS = dict(
    min_cluster_size=MIN_CLUSTER_SIZE,
    min_samples=3,
    cluster_selection_method="eom",
    cluster_selection_epsilon=0.02,
)

TOPONYMY_LAYER_INDEX = 0
TOPONYMY_CLUSTER_PARAMS = dict(
    min_clusters=5,
    min_samples=3,
    base_min_cluster_size=MIN_CLUSTER_SIZE,
)


Dataset: 382 documents → min_cluster_size=10


In [10]:
print(f"Topic backend: {TOPIC_BACKEND}")
print("Clustering configuration is applied directly in the selected topic backend fit function.")


Topic backend: toponymy_evoc
Clustering configuration is applied directly in the selected topic backend fit function.


### 5.5 Topic Modeling Configuration

In [11]:
# --- TOPIC MODELING / LLM LABELING ---
# LLM Providers: 'local', 'huggingface_api', 'openrouter'

LLM_PROVIDER = "openrouter"
LLM_MODEL = "google/gemini-3-flash-preview"  # or an OpenRouter model
LLM_API_KEY = os.getenv("OPENROUTER_API_KEY")

# Representation pipeline (BERTopic only)
PIPELINE_MODELS = ["POS", "KeyBERT", "MMR"]
PARALLEL_MODELS = ["MMR", "POS", "KeyBERT"]

# Toponymy text embedder model (OpenRouter/OpenAI-compatible endpoint)
TOPONYMY_EMBEDDING_MODEL = EMBEDDING_MODEL

# min_df for CountVectorizer: use 1 for small samples, 2+ for full data
MIN_DF = 1 if (SAMPLE_SIZE and SAMPLE_SIZE < 1000) else 2


In [12]:
from ads_bib.topic_model import fit_bertopic, fit_toponymy, reduce_outliers, build_topic_dataframe
import numpy as np

if TOPIC_BACKEND == "bertopic":
    topic_model = fit_bertopic(
        documents,
        reduced_5d,
        llm_provider=LLM_PROVIDER,
        llm_model=LLM_MODEL,
        pipeline_models=PIPELINE_MODELS,
        parallel_models=PARALLEL_MODELS,
        min_df=MIN_DF,
        clustering_method=CLUSTERING_METHOD,
        clustering_params=CLUSTER_PARAMS,
        api_key=LLM_API_KEY,
        openrouter_cost_mode=OPENROUTER_COST_MODE,
        cost_tracker=tracker,
    )

    topics = np.array(topic_model.topics_)
    topics = reduce_outliers(
        topic_model,
        documents,
        topics,
        reduced_5d,
        threshold=0.8,
        llm_provider=LLM_PROVIDER,
        llm_model=LLM_MODEL,
        api_key=LLM_API_KEY,
        openrouter_cost_mode=OPENROUTER_COST_MODE,
        cost_tracker=tracker,
    )
    topic_info = topic_model.get_topic_info()

elif TOPIC_BACKEND in {"toponymy", "toponymy_evoc"}:
    topic_model, topics, topic_info = fit_toponymy(
        documents,
        embeddings,
        reduced_5d,
        backend=TOPIC_BACKEND,
        layer_index=TOPONYMY_LAYER_INDEX,
        llm_provider=LLM_PROVIDER,
        llm_model=LLM_MODEL,
        embedding_model=TOPONYMY_EMBEDDING_MODEL,
        api_key=LLM_API_KEY,
        openrouter_cost_mode=OPENROUTER_COST_MODE,
        clusterer_params=TOPONYMY_CLUSTER_PARAMS,
        cost_tracker=tracker,
    )

else:
    raise ValueError(
        f"Invalid TOPIC_BACKEND '{TOPIC_BACKEND}'. "
        "Use 'bertopic', 'toponymy', or 'toponymy_evoc'."
    )

df = build_topic_dataframe(
    df,
    topic_model,
    topics,
    reduced_2d,
    embeddings,
    topic_info=topic_info,
)
topic_info


Fitting Toponymy backend='toponymy_evoc' (LLM: openrouter/google/gemini-3-flash-preview) ...


Selecting facility_location exemplars:   0%|          | 0/13 [00:00<?, ?cluster/s]

embedding texts:   0%|          | 0/27 [00:00<?, ?it/s]

Building topic names by layer:   0%|          | 0/1 [00:00<?, ?layer/s]

Generating informative keyphrases:   0%|          | 0/13 [00:00<?, ?cluster/s]

Generating prompts for layer 0:   0%|          | 0/13 [00:00<?, ?topic/s]

Generating topic names for layer 0:   0%|          | 0/13 [00:00<?, ?topic/s]

  Selected Toponymy layer 0/0: 13 topics, 128 outliers.
  OpenRouter cost: $0.0072 (14/14 priced; direct=14, fetched=0, mode=hybrid)


,Topic,Name,Main
0,-1,Outlier Topic,Outlier Topic
1,0,"Fundamental Physical Constants, Quantum Cosmol...","Fundamental Physical Constants, Quantum Cosmol..."
2,1,"Foundational Principles of Quantum Mechanics, ...","Foundational Principles of Quantum Mechanics, ..."
3,2,Relativistic Gravitational Effects on Light Pr...,Relativistic Gravitational Effects on Light Pr...
4,3,General Relativistic Generalizations of Ertel'...,General Relativistic Generalizations of Ertel'...
5,4,Historical and Theoretical Intersections of Fu...,Historical and Theoretical Intersections of Fu...
6,5,Historical and Theoretical Analysis of Post-Ne...,Historical and Theoretical Analysis of Post-Ne...
7,6,Analytical Formulations of the Mach-Einstein D...,Analytical Formulations of the Mach-Einstein D...
8,7,"Singularities, Shock Waves, and Particle Model...","Singularities, Shock Waves, and Particle Model..."
9,8,"Biographical Tributes, Obituaries, and Histori...","Biographical Tributes, Obituaries, and Histori..."


### 5.6 Visualize Topics

In [13]:
from ads_bib.visualize import create_topic_map

plot = create_topic_map(
    df,
    label_column="Name",
    title="ADS Bibliometric Map",
    subtitle=f"Topics labeled with {LLM_PROVIDER}/{LLM_MODEL}",
    dark_mode=True,
    output_path=paths["output"] / "topic_map.html",
)
# plot  # uncomment to display inline

Saved: topic_map.html


### 5.7 Curate Dataset

Review the cluster summary, then remove clusters that don't fit your research question.

In [ ]:
from ads_bib.curate import get_cluster_summary, remove_clusters

get_cluster_summary(df)

In [ ]:
# Remove unwanted clusters (set IDs based on the summary above)
CLUSTERS_TO_REMOVE = []  # e.g. [3, 7, -1]

if CLUSTERS_TO_REMOVE:
    df = remove_clusters(df, CLUSTERS_TO_REMOVE)

### Checkpoint: Save Curated Dataset

In [ ]:
from ads_bib._utils.io import save_parquet

# Ensure References is list type for Parquet
df["References"] = df["References"].apply(lambda x: x if isinstance(x, list) else [])

save_parquet(df, paths["processed"] / "curated_dataset.parquet")
print(f"Curated dataset: {len(df):,} documents")

In [ ]:
from ads_bib._utils.io import load_parquet

df = load_parquet(paths["processed"] / "curated_dataset.parquet")
df.head(10)

---
# Phase 6: Citation Networks

Compute citation networks from the curated dataset and export
to GEXF (Gephi), Graphology JSON (Sigma.js), CSV, and/or Web of Science format.

### 6.1 Citation Configuration

In [ ]:
# --- CITATION CONFIGURATION ---
METRICS = ["direct", "co_citation", "bibliographic_coupling", "author_co_citation"]
MIN_COUNTS = {
    "direct": 1,
    "co_citation": 1,
    "bibliographic_coupling": 1,
    "author_co_citation": 1,
}
AUTHORS_FILTER = None  # e.g. ['Treder', 'Kreisel'] or None for all
OUTPUT_FORMAT = "gexf"  # Options: 'gexf', 'graphology', 'csv', 'all'

### 6.2 Build Node Table & Compute Networks

In [ ]:
from ads_bib.citations import build_all_nodes, process_all_citations, export_wos_format

# Reload processed data if needed
all_nodes = build_all_nodes(publications, refs)

results = process_all_citations(
    bibcodes, references, publications, refs, all_nodes,
    metrics=METRICS,
    min_counts=MIN_COUNTS,
    authors_filter=AUTHORS_FILTER,
    output_format=OUTPUT_FORMAT,
    output_dir=paths["output"],
)

### 6.3 Export Web of Science Format

In [ ]:
suffix = "_filtered" if AUTHORS_FILTER else ""
export_wos_format(
    publications, refs,
    output_path=paths["output"] / f"download_wos_export{suffix}.txt",
)

---
# Summary

Final dataset statistics and output file listing.

In [ ]:
import os

print("="*60)
print("PIPELINE COMPLETE")
print("="*60)
print(f"Publications:     {len(publications):,}")
print(f"References:       {len(refs):,}")
print(f"Curated dataset:  {len(df):,}")
print(f"Topics found:     {df['topic_id'].nunique()}")
print()
print("Output files:")
for root, dirs, files in os.walk(paths["output"]):
    for f in sorted(files):
        fpath = Path(root) / f
        size_mb = fpath.stat().st_size / 1024 / 1024
        print(f"  {fpath.relative_to(paths['output'])} ({size_mb:.1f} MB)")

print()
print(tracker.compact_summary())
